# AI-Based Analysis of Gene and DNA Sequence Patterns
### **Course:** Introduction to Biological Data (Amrita Vishwa Vidyapeetham, Coimbatore)
### **Department:** Artificial Intelligence and Data Science (1st Year)
### **Role:** Person 3 — AI/ML Model Development & Evaluation
### **Lead:** Harshith Krishna (CB.AI.U4AID26023)
---
## 1. Project Overview & Role Objectives
In eukaryotic genomes, precursor messenger RNA (pre-mRNA) undergoes **splicing**, where non-coding sequences (**introns**) are excised and coding sequences (**exons**) are ligated together to form mature mRNA. Identifying splice junctions computationally is vital for **genome annotation**, **gene function discovery**, and **identifying disease-causing splice mutations**.

As **Person 3**, our core responsibilities are:
1. **Establish Baseline Models:** Implement Logistic Regression and Naive Bayes.
2. **Evaluate Diverse ML Architectures:** Train and compare Support Vector Machines (SVM), Random Forests, XGBoost, and Multi-Layer Perceptrons (MLP).
3. **Prevent Data Leakage & Overfitting:** Utilize 5-Fold Stratified Cross-Validation on the training split.
4. **Multi-Metric Holdout Evaluation:** Assess Accuracy, Balanced Accuracy, Macro/Weighted F1, and Multi-Class ROC-AUC (OvR).
5. **Hyperparameter Tuning:** Systematically optimize the best ensemble architectures.
6. **Biological Interpretability:** Extract feature importances and validate them against known molecular biology (donor GT consensus, acceptor AG consensus, and polypyrimidine tracts).
7. **Inference Engine:** Deliver a reusable prediction function for real-time inference and Streamlit integration.

In [ ]:
# Import required libraries
import os
import sys
import time
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, auc
)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

sns.set_theme(style='whitegrid', font_scale=1.0)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print('All libraries imported successfully!')

## 2. Load Processed Datasets (Handshake with Person 2)
We load `X_train.csv`, `X_test.csv`, `y_train.csv`, and `y_test.csv` prepared by Person 2.
- Total sequence instances: 3,178 cleaned records.
- Train split (80%): 2,542 samples.
- Test split (20%): 636 samples.
- Feature dimensionality: 91 features (mononucleotide counts/frequencies, GC content, sequence length, ambiguity count, 16 dinucleotides, 64 trinucleotides).

In [ ]:
base_dir = Path('..')
data_dir = base_dir / 'data' / 'processed'

X_train = pd.read_csv(data_dir / 'X_train.csv')
X_test = pd.read_csv(data_dir / 'X_test.csv')
y_train_raw = pd.read_csv(data_dir / 'y_train.csv').iloc[:, 0].str.strip()
y_test_raw = pd.read_csv(data_dir / 'y_test.csv').iloc[:, 0].str.strip()

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)
y_test = label_encoder.transform(y_test_raw)

classes = list(label_encoder.classes_)
print(f'X_train shape: {X_train.shape} | y_train shape: {y_train.shape}')
print(f'X_test shape:  {X_test.shape}  | y_test shape:  {y_test.shape}')
print('Target Classes:', dict(zip(classes, range(len(classes)))))
print('\nClass Distribution in Train:')
print(pd.Series(y_train_raw).value_counts())
print('\nClass Distribution in Test:')
print(pd.Series(y_test_raw).value_counts())

## 3. Candidate Model Suite & Pipeline Construction
We define 6 diverse machine learning architectures to systematically test linear, probabilistic, kernel-based, tree ensemble, boosted, and neural representations:
1. **Logistic Regression (Baseline):** Multinomial linear model with L2 penalty and balanced class weighting.
2. **Gaussian Naive Bayes:** Generative probabilistic classifier assuming feature conditional independence.
3. **Support Vector Machine (RBF):** Kernelized margin classifier finding non-linear decision boundaries.
4. **Random Forest:** Bagged ensemble of 250 decision trees, capturing complex non-linear feature interactions.
5. **XGBoost:** Gradient boosted decision trees using second-order gradients and regularization.
6. **Multi-Layer Perceptron (MLP):** Feed-forward neural network with ReLU activations and Adam optimizer.

*Note: Standard scaling is wrapped into `sklearn.pipeline.Pipeline` to guarantee strict zero-leakage cross-validation.*

In [ ]:
models = {
    'Logistic Regression (Baseline)': Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_SEED))
    ]),
    'Gaussian Naive Bayes': Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', GaussianNB())
    ]),
    'Support Vector Machine (RBF)': Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', SVC(kernel='rbf', C=1.5, gamma='scale', probability=True, class_weight='balanced', random_state=RANDOM_SEED))
    ]),
    'Random Forest': RandomForestClassifier(
        n_estimators=250, max_depth=16, min_samples_split=4, min_samples_leaf=2,
        class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1
    ),
    'XGBoost Classifier': XGBClassifier(
        n_estimators=250, max_depth=5, learning_rate=0.08, subsample=0.85, colsample_bytree=0.85,
        eval_metric='mlogloss', random_state=RANDOM_SEED, n_jobs=-1
    ),
    'Multi-Layer Perceptron (MLP)': Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', MLPClassifier(hidden_layer_sizes=(128, 64), activation='relu', solver='adam', alpha=0.001, max_iter=400, early_stopping=True, random_state=RANDOM_SEED))
    ])
}
print(f'Initialized {len(models)} candidate models.')

## 4. Stratified 5-Fold Cross-Validation
Stratified 5-Fold Cross-Validation partitions `X_train` into 5 folds while preserving the relative proportions of EI, IE, and N classes across each split. This tests model stability and prevents overfitting.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_results = {}

for name, model in models.items():
    t0 = time.time()
    scores = cross_validate(
        model, X_train, y_train, cv=cv,
        scoring={'acc': 'accuracy', 'f1_macro': 'f1_macro'}, n_jobs=-1
    )
    elapsed = time.time() - t0
    cv_results[name] = {
        'cv_acc_mean': scores['test_acc'].mean() * 100,
        'cv_acc_std': scores['test_acc'].std() * 100,
        'cv_f1_mean': scores['test_f1_macro'].mean(),
        'cv_f1_std': scores['test_f1_macro'].std(),
        'time_sec': elapsed
    }
    print(f"[{name}] CV Acc: {cv_results[name]['cv_acc_mean']:.2f}% (±{cv_results[name]['cv_acc_std']:.2f}%) | Macro F1: {cv_results[name]['cv_f1_mean']:.4f}")

## 5. Holdout Test Set Evaluation & Comparative Benchmark
Each model is trained on the full training set and evaluated on the unseen test set `X_test`. We log Accuracy, Balanced Accuracy, Macro Precision/Recall/F1, and Multi-Class ROC-AUC (One-vs-Rest).

In [ ]:
benchmark_list = []
fitted_models = {}
predictions = {}
probabilities = {}

for name, model in models.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test) if hasattr(model, 'predict_proba') else None
    
    fitted_models[name] = model
    predictions[name] = y_pred
    probabilities[name] = y_prob
    
    acc = accuracy_score(y_test, y_pred)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    weighted_f1 = f1_score(y_test, y_pred, average='weighted')
    roc_auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='macro') if y_prob is not None else np.nan
    
    benchmark_list.append({
        'Model': name,
        'CV Acc (%)': round(cv_results[name]['cv_acc_mean'], 2),
        'Test Acc (%)': round(acc * 100, 2),
        'Balanced Acc (%)': round(bal_acc * 100, 2),
        'Macro F1': round(macro_f1, 4),
        'Weighted F1': round(weighted_f1, 4),
        'ROC-AUC (OVR)': round(roc_auc, 4),
        'Train Time (s)': round(train_time, 2)
    })

benchmark_df = pd.DataFrame(benchmark_list).sort_values(by='Macro F1', ascending=False).reset_index(drop=True)
print(benchmark_df.to_string())

## 6. Confusion Matrix Analysis
Visualizing confusion matrices across all candidate models to inspect error distributions between Exon-Intron (EI), Intron-Exon (IE), and Neither (N).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, (name, y_pred) in enumerate(predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    
    annot = np.empty_like(cm).astype(str)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            annot[i, j] = f'{cm[i, j]}\n({cm_norm[i, j]:.1f}%)'
            
    sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', cbar=False, ax=axes[idx],
                xticklabels=classes, yticklabels=classes, annot_kws={'size': 10, 'weight': 'bold'})
    acc_val = accuracy_score(y_test, y_pred) * 100
    f1_val = f1_score(y_test, y_pred, average='macro')
    axes[idx].set_title(f'{name}\nAcc: {acc_val:.1f}% | Macro F1: {f1_val:.3f}', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Predicted Class', fontweight='bold')
    axes[idx].set_ylabel('True Class', fontweight='bold')

plt.suptitle('DNA Splice Junction Classification: Confusion Matrix Across Models', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7. Biological Interpretability: Feature Importance Analysis
Rather than deploying a black-box model, we extract tree feature importances (Gini gain) from Random Forest and XGBoost. We inspect whether the top predictive k-mers correspond to known molecular biology:
- **Donor Splice Site (EI):** Governed by the canonical 5' `GT` dinucleotide consensus.
- **Acceptor Splice Site (IE):** Governed by the 3' `AG` dinucleotide consensus preceded by a pyrimidine-rich tract (high `C` and `T` content).
- **Non-Splice (N):** Characterized by broad sequence distributions without localized consensus motifs.

In [ ]:
rf = fitted_models['Random Forest']
xgb = fitted_models['XGBoost Classifier']

imp_df = pd.DataFrame({
    'Feature': X_train.columns,
    'RandomForest': rf.feature_importances_,
    'XGBoost': xgb.feature_importances_
})
imp_df['Mean_Importance'] = imp_df[['RandomForest', 'XGBoost']].mean(axis=1)
imp_df = imp_df.sort_values(by='Mean_Importance', ascending=False).reset_index(drop=True)

print('Top 15 Most Important DNA Features:')
print(imp_df.head(15))

plt.figure(figsize=(10, 6))
sns.barplot(data=imp_df.head(15), y='Feature', x='Mean_Importance', palette='viridis')
plt.title('Top 15 Predictive Sequence Motifs for Splice-Junction Recognition', fontsize=13, fontweight='bold')
plt.xlabel('Relative Importance Score', fontweight='bold')
plt.ylabel('Genomic Feature / k-mer Motif', fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Real-Time Inference Demo (Testing Arbitrary DNA Sequences)
We test the production inference engine on sample sequences to demonstrate biological pattern classification.

In [ ]:
sys.path.append(str(base_dir / 'src' / 'models'))
from predict import DNAPatternPredictor

predictor = DNAPatternPredictor(models_dir=base_dir / 'models_saved')

# Test sequences
test_sample = 'CCAGCTGCATCACAGGAGGCCAGCGAGCAGGTCTGTTCCAAGGGCCTTCGAGCCAGTCTG'
res = predictor.predict(test_sample)

print(f"Predicted Class: {res['predicted_label']} - {res['predicted_name']}")
print(f"Confidence:      {res['confidence_percent']}%")
print(f"Class Probs:     {res['class_probabilities']}")
print(f"Biological Role: {res['biological_role']}")
print(f"Consensus Motif: {res['biological_consensus']}")